# 03 — Feature Engineering

**Input:** `../data/processed/pm_day_clean.csv`, `../data/processed/embeddings.npy`
**Output:** `../data/processed/pm_day_features.csv`

**Description:**
- Compute PCA on embeddings (content representation)
- Compute engagement features (word count, log word count, minimal text flag)
- Compute within-person instability (cosine distance to person centroid)
- Decompose all features into within-person and between-person components
- Save feature-enriched dataset for modeling

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_clean.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")

PID_COL = "expiwell_id_clean"
TEXT_COL = "pm_day_text"
CRISIS_COL = "crisis_PM_from_full"

N_PCS = 20
N_PCS_CONTENT = 5
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [ ]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)

assert X_text.shape[0] == len(pm_day), \
    f"Embedding rows ({X_text.shape[0]}) != data rows ({len(pm_day)})"

print("Loaded data:", pm_day.shape)
print("Loaded embeddings:", X_text.shape)

In [ ]:
# =========================
# ENGAGEMENT FEATURES
# =========================
txt = pm_day[TEXT_COL].fillna("").astype(str)
pm_day["word_count"] = txt.str.split().map(len).astype(float)
pm_day["log1p_wc"] = np.log1p(pm_day["word_count"])
pm_day["wc_le1"] = (pm_day["word_count"] <= 1).astype(float)

print("Engagement features computed.")
print("  Mean word count:", pm_day["word_count"].mean().round(1))
print("  % minimal text (<=1 word):", pm_day["wc_le1"].mean().round(3))

In [ ]:
# =========================
# PCA ON EMBEDDINGS
# =========================
def l2_normalize_rows(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)

X_norm = l2_normalize_rows(X_text)

pca_full = PCA(n_components=N_PCS, random_state=RANDOM_SEED)
X_pcs = pca_full.fit_transform(X_text)

for k in range(N_PCS):
    pm_day[f"PC{k+1}"] = X_pcs[:, k]

print(f"PCA: {N_PCS} components")
print("  Variance explained (first 5):", np.round(pca_full.explained_variance_ratio_[:5], 4))
print("  Cumulative variance ({} PCs): {:.3f}".format(N_PCS, pca_full.explained_variance_ratio_.sum()))

In [ ]:
# =========================
# WITHIN-PERSON INSTABILITY
# =========================
pid = pm_day[PID_COL].astype(str).values
centroids = np.zeros_like(X_norm, dtype=float)

for p in np.unique(pid):
    idx = np.where(pid == p)[0]
    c = X_norm[idx].mean(axis=0)
    nrm = np.linalg.norm(c)
    if nrm > 0:
        c = c / nrm
    centroids[idx] = c

pm_day["instability_cosdist"] = 1.0 - np.sum(X_norm * centroids, axis=1)

print("Instability descriptives:")
print(pm_day["instability_cosdist"].describe())

In [ ]:
# =========================
# WITHIN vs BETWEEN DECOMPOSITION
# =========================
MECH_COLS = (
    ["log1p_wc", "wc_le1", "instability_cosdist"] +
    [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]
)

g = pm_day.groupby(PID_COL)
for c in MECH_COLS:
    pm_day[f"{c}_between"] = g[c].transform("mean")
    pm_day[f"{c}_within"] = pm_day[c] - pm_day[f"{c}_between"]

print("Within/between decomposition computed for:", MECH_COLS)

In [ ]:
# =========================
# SAVE
# =========================
pm_day.to_csv(OUT_PATH, index=False)
print("Saved feature dataset:", OUT_PATH)
print("Shape:", pm_day.shape)